In [10]:
from pathlib import Path

docs_path = Path("../docs")
file_path = docs_path / "real_10k_example.txt"

text = file_path.read_text()

print(text[:500])

Apple 10-K Risk Factors
Microsoft 10-K Risk Factors



In [11]:
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks


chunks = chunk_text(text)

len(chunks), chunks[0]

(1, 'Apple 10-K Risk Factors\nMicrosoft 10-K Risk Factors\n')

In [13]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../chroma_db")

collection = chroma_client.get_or_create_collection(
    name="sec_filings"
)

In [14]:
for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk],
        ids=[f"sample_10k_chunk_{i}"]
    )

print("Chunks stored in ChromaDB")

Chunks stored in ChromaDB


In [15]:
query = "What risks affect technology stocks and bonds?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

retrieved_context = "\n\n".join(results["documents"][0])

print(retrieved_context)

The company faces risks related to market volatility, interest rates, inflation, cybersecurity threats, supply chain disruption, competition, regulatory changes, and macroeconomic uncertainty.

Technology companies may experience higher volatility due to valuation changes, innovation cycles, and sen

e to valuation changes, innovation cycles, and sensitivity to interest rates.

Bond investments may decline when interest rates rise because bond prices generally move inversely to interest rates.

Gold may behave differently from equities and bonds because investors often view it as a defensive ass

because investors often view it as a defensive asset during periods of uncertainty.



In [16]:
rag_prompt = f"""
You are a financial risk analyst.

Use the retrieved SEC-style context below to explain the risk environment.

Question:
{query}

Retrieved Context:
{retrieved_context}

Explain in plain English:
- key risks
- why they matter
- how they relate to market volatility
- how this improves the earlier risk analysis
"""

In [17]:
from openai import OpenAI
from dotenv import load_dotenv

import os

load_dotenv("../.env")

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [18]:
response = client.responses.create(
    model="gpt-5.5",
    input=rag_prompt
)

rag_analysis = response.output_text

print(rag_analysis)

Technology stocks and bonds are exposed to different but connected risks. Together, they can make a portfolio more sensitive to market volatility, interest-rate changes, and broader economic uncertainty.

## Key risks

### 1. Market volatility
Technology stocks can move sharply when investor expectations change. Their valuations often depend on future growth, so even small changes in earnings expectations, interest rates, or investor sentiment can cause large price swings.

Bonds can also be volatile, especially when interest rates move quickly. When rates rise, existing bonds with lower coupons usually become less attractive, causing their prices to fall.

### 2. Interest-rate risk
This is a major risk for both technology stocks and bonds.

For technology stocks, higher interest rates reduce the present value of future earnings. Since many technology companies are valued based on expected future growth, rising rates can pressure stock prices.

For bonds, the relationship is more direc